
## Final application notebook

This version keeps the successful structured extraction from the previous version and makes the **Key Points classifier stricter**.

### Output sections

1. Key Points — factual observations/findings only
2. Decisions — explicit finalized decisions only
3. Action Items — explicit first-person commitments only
4. Important Dates / Deadlines — scheduled events and explicit dates

### Key Point rule

A statement is rejected from Key Points if it describes:
- a decision,
- an action/commitment,
- a request,
- a question,
- a schedule/deadline,
- a requirement or planning statement such as `needs to`, `need to`, `should`, `must`, or `has to`.

This prevents a planning statement such as:

`The user guide needs the export workflow and filter examples.`

from appearing as a Key Point when the actual action belongs under Action Items.

FLAN-T5 is used only after the factual source statement has passed these filters. Generated text is validated and falls back to the source statement if meaning changes.

No new folders are created. Existing matching summary files are overwritten. `dataset/` is untouched.


In [1]:
# Load the FLAN-T5 tokenizer

import os
import re
import glob
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"
MODEL_FILE = "flan_t5_model.pt"

TRANSCRIPT_DIR = os.path.join(
    "outputs", "assemblyai", "transcripts"
)

SUMMARY_DIR = os.path.join(
    "outputs", "Flan-T5"
)

MAX_KEY_POINTS = 5
MAX_DECISIONS = 5
MAX_ACTIONS = 8
MAX_DATES = 8

print("Transcript folder:", os.path.abspath(TRANSCRIPT_DIR))
print("Summary folder:", os.path.abspath(SUMMARY_DIR))
print("Model:", os.path.abspath(MODEL_FILE))


c:\Users\dell\anaconda3\envs\tf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transcript folder: c:\Users\dell\OneDrive\Desktop\DL\AI-Meeting-Minutes-Generator\outputs\assemblyai\transcripts
Summary folder: c:\Users\dell\OneDrive\Desktop\DL\AI-Meeting-Minutes-Generator\outputs\Flan-T5
Model: c:\Users\dell\OneDrive\Desktop\DL\AI-Meeting-Minutes-Generator\flan_t5_model.pt


In [2]:
if not os.path.isfile(MODEL_FILE):
    raise FileNotFoundError(
        f"Model file not found: {os.path.abspath(MODEL_FILE)}"
    )

if not os.path.isdir(TRANSCRIPT_DIR):
    raise FileNotFoundError(
        f"Transcript folder not found: {os.path.abspath(TRANSCRIPT_DIR)}"
    )

if not os.path.isdir(SUMMARY_DIR):
    raise FileNotFoundError(
        f"Summary folder not found: {os.path.abspath(SUMMARY_DIR)}"
    )

transcript_files = sorted(
    glob.glob(
        os.path.join(TRANSCRIPT_DIR, "*.txt")
    )
)

if not transcript_files:
    raise FileNotFoundError(
        "No transcript TXT files were found."
    )

print("Found", len(transcript_files), "transcript(s).")


Found 11 transcript(s).


In [3]:
# Load the FLAN-T5 tokenizer

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

state_dict = torch.load(
    MODEL_FILE,
    map_location=device
)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

print("Fine-tuned FLAN-T5 loaded successfully.")
print("Device:", device)


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5917.65it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fine-tuned FLAN-T5 loaded successfully.
Device: cpu


In [4]:
# Clean transcript text

def clean(text):
    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def normalize(text):
    return re.sub(
        r"[^a-z0-9 ]",
        " ",
        clean(text).lower()
    ).strip()


def similarity(a, b):
    aa = set(normalize(a).split())
    bb = set(normalize(b).split())

    if not aa or not bb:
        return 0.0

    return len(aa & bb) / len(aa | bb)


def dedupe(items, threshold=0.68):
    result = []

    for item in items:
        item = clean(item)

        if not item:
            continue

        if any(
            similarity(item, old) >= threshold
            for old in result
        ):
            continue

        result.append(item)

    return result


def parse_transcript(text):
    records = []

    for raw_line in text.splitlines():
        line = clean(raw_line)

        if not line:
            continue

        match = re.match(
            r"^([^:]{1,70}):\s*(.+)$",
            line
        )

        if match:
            speaker = clean(match.group(1))
            body = clean(match.group(2))
        else:
            speaker = ""
            body = line

        if len(body.split()) >= 4:
            records.append({
                "speaker": speaker,
                "body": body
            })

    return records


In [5]:
# Define language patterns for meeting information

QUESTION_RE = re.compile(
    r"\?$|^(?:can|could|would|should|shall|do|does|did|"
    r"is|are|will|what|when|where|why|how|may|might)\b",
    re.I
)

COMMITMENT_RE = re.compile(
    r"^(?:I\s+(?:will|shall|can|am going to|plan to|intend to)\b|I['’]ll\b)",
    re.I
)

REQUEST_RE = re.compile(
    r"^(?:please|can you|could you|would you|let's|lets)\b",
    re.I
)

DECISION_RE = re.compile(
    r"^(?:we\s+(?:have\s+)?(?:decided|agreed|approved|confirmed|finalized)\b|"
    r"the\s+(?:decision|plan|approach)\s+(?:is|was)\b|"
    r"we['’]ll\s+(?:use|adopt|keep|go with|proceed with)\b|"
    r"we\s+will\s+(?:use|adopt|keep|go with|proceed with)\b)",
    re.I
)

REQUIREMENT_RE = re.compile(
    r"\b(?:should|needs?\s+to|must|has\s+to|have\s+to|need\s+to|"
    r"is\s+required\s+to|are\s+required\s+to|required\s+to)\b",
    re.I
)

def is_question(text):
    return bool(QUESTION_RE.search(clean(text)))

def is_commitment(text):
    return bool(COMMITMENT_RE.match(clean(text)))

def is_request(text):
    return bool(REQUEST_RE.match(clean(text)))

def is_decision(text):
    return bool(DECISION_RE.match(clean(text)))


In [6]:
# Extract important dates and deadlines

DATE_RE = re.compile(
    r"\b(?:monday|tuesday|wednesday|thursday|friday|saturday|sunday)"
    r"\s+(?:at\s+\d{1,2}(?::\d{2})?\s*(?:am|pm)|"
    r"(?:morning|afternoon|evening))\b|"
    r"\bnext\s+(?:monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b|"
    r"\b(?:today|tomorrow|yesterday)"
    r"(?:\s+(?:morning|afternoon|evening))?\b|"
    r"\b(?:this|next|coming)\s+(?:week|month)\b|"
    r"\b\d{1,2}[/-]\d{1,2}(?:[/-]\d{2,4})?\b|"
    r"\b(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|"
    r"jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|"
    r"nov(?:ember)?|dec(?:ember)?)\s+\d{1,2}(?:,\s*\d{4})?\b",
    re.I
)

DATE_CONTEXT_RE = re.compile(
    r"\b(?:by|before|until|on|at|scheduled|deadline|due|target|"
    r"review|meeting|release|test|demo|presentation|submission|"
    r"delivery|launch|call|appointment)\b",
    re.I
)

def extract_dates(records):
    items = []
    seen = set()

    for record in records:
        body = clean(record["body"])

        for match in DATE_RE.finditer(body):
            phrase = clean(match.group(0))

            if not DATE_CONTEXT_RE.search(body):
                continue

            before = clean(body[:match.start()].strip(" ,:-"))
            if not before:
                label = "Meeting date"
            else:
                words = before.split()
                label = " ".join(words[-10:])

            item = f"{label} — {phrase}."
            key = normalize(item)

            if key in seen:
                continue

            seen.add(key)
            items.append(item)

    return dedupe(items, threshold=0.82)[:MAX_DATES]


In [7]:
# Extract assigned action items

def strip_commitment(body):
    body = clean(body)
    body = re.sub(
        r"^I\s+(?:will|shall|can|am going to|plan to|intend to)\s+",
        "",
        body,
        flags=re.I
    )
    body = re.sub(r"^I['’]ll\s+", "", body, flags=re.I)
    return clean(body).rstrip("?.! ")

def extract_actions(records):
    actions = []
    seen = set()

    for record in records:
        body = clean(record["body"])

        if not body or is_question(body) or is_request(body):
            continue

        if not is_commitment(body):
            continue

        task = strip_commitment(body)

        if len(task.split()) < 3:
            continue

        speaker = clean(record["speaker"]) or "Team"
        key = normalize(f"{speaker} {task}")

        if key in seen:
            continue

        seen.add(key)
        actions.append(f"{speaker} — {task}")

    return dedupe(actions, threshold=0.84)[:MAX_ACTIONS]


In [8]:
# Extract confirmed decisions

def clean_decision(body):
    body = clean(body)
    body = re.sub(
        r"^the\s+(?:decision|plan|approach)\s+(?:is|was)\s+(?:that\s+)?",
        "",
        body,
        flags=re.I
    )
    body = re.sub(
        r"^we\s+(?:have\s+)?(?:decided|agreed|approved|confirmed|finalized)\s+(?:that\s+)?",
        "",
        body,
        flags=re.I
    )
    return clean(body).rstrip("?.! ")

def extract_decisions(records):
    decisions = []
    seen = set()

    for record in records:
        body = clean(record["body"])

        if not body or is_question(body) or is_request(body):
            continue

        if not is_decision(body):
            continue

        decision = clean_decision(body)

        if len(decision.split()) < 4:
            continue

        key = normalize(decision)

        if key in seen:
            continue

        seen.add(key)
        decisions.append(decision)

    return dedupe(decisions, threshold=0.78)[:MAX_DECISIONS]


In [9]:
# Select factual key points from the transcript

def is_factual_keypoint(body):
    text = clean(body)
    lower = text.lower()

    if len(text.split()) < 6:
        return False

    if is_question(text) or is_request(text) or is_commitment(text) or is_decision(text):
        return False

    if REQUIREMENT_RE.search(text):
        return False

    planning_starts = (
        "we should", "we need", "we have to", "we must",
        "let's", "lets", "please", "can we", "could we",
        "i think we should"
    )
    if lower.startswith(planning_starts):
        return False

    conversational_starts = (
        "i agree", "that makes sense", "sounds good",
        "okay", "ok", "right", "yes", "no",
        "thanks", "thank you"
    )
    if lower.startswith(conversational_starts):
        return False

    return True

def keypoint_score(body, frequency):
    words = normalize(body).split()
    informative = sum(
        1 for word in words
        if len(word) >= 4 and word not in {
            "this", "that", "with", "from", "have", "will",
            "were", "been", "there", "their", "about"
        }
    )
    numbers = sum(1 for word in words if any(ch.isdigit() for ch in word))
    return (
        min(len(words), 30) * 0.8
        + informative * 1.2
        + numbers * 2.0
        + min(frequency.get(normalize(body), 1), 3)
    )

def select_keypoint_sources(records, decisions):
    frequency = {}

    for record in records:
        body = clean(record["body"])
        key = normalize(body)
        if key:
            frequency[key] = frequency.get(key, 0) + 1

    candidates = []

    for record in records:
        body = clean(record["body"])

        if not is_factual_keypoint(body):
            continue

        if any(similarity(body, decision) >= 0.55 for decision in decisions):
            continue

        candidates.append(body)

    candidates = dedupe(candidates, threshold=0.68)
    candidates.sort(
        key=lambda text: keypoint_score(text, frequency),
        reverse=True
    )

    selected = []

    for candidate in candidates:
        if any(similarity(candidate, old) >= 0.55 for old in selected):
            continue

        selected.append(candidate)

        if len(selected) >= MAX_KEY_POINTS:
            break

    return selected

def extract_key_points(records, decisions):
    sources = select_keypoint_sources(records, decisions)
    return [flan_keypoint(source) for source in sources]


In [10]:
# Format factual key points conservatively

def normalize_keypoint_output(text):
    text = clean(text).strip("•-* ")
    if not text:
        return text

    numeric_start = re.match(
        r"^(\d+(?:[.,]\d+)*)(\s+)(.+)$",
        text
    )

    if numeric_start:
        number, spacing, rest = numeric_start.groups()
        rest = rest[:1].lower() + rest[1:] if rest else rest
        return number + spacing + rest

    for i, ch in enumerate(text):
        if ch.isalpha():
            return text[:i] + ch.upper() + text[i + 1:]

    return text

def flan_keypoint(statement):
    statement = normalize_keypoint_output(statement)

    if len(statement.split()) <= 18:
        return statement

    prompt = (
        "Rewrite this factual meeting statement as ONE concise key point. "
        "Preserve the exact subject and object. "
        "Preserve who did what and all important numbers. "
        "Do not turn it into a decision, recommendation, action item, "
        "request, question, date or deadline. "
        "Do not add information. "
        "Return one sentence only.\n\n"
        f"Statement: {statement}"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=45,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    result = clean(
        tokenizer.decode(
            output[0],
            skip_special_tokens=True
        )
    ).strip("•-* ")

    if not result:
        return statement

    forbidden = (
        "i agree", "we should", "we need", "let's", "lets",
        "can you", "could you", "please", "decision",
        "action item", "deadline", "meeting to discuss"
    )

    if any(phrase in result.lower() for phrase in forbidden):
        return statement

    if similarity(result, statement) < 0.20:
        return statement

    source_words = set(normalize(statement).split())
    generated_words = set(normalize(result).split())

    important_numbers = {
        word for word in source_words
        if any(ch.isdigit() for ch in word)
    }

    if important_numbers and not important_numbers.issubset(generated_words):
        return statement

    return normalize_keypoint_output(
        result.rstrip("?.! ") + "."
    )


In [11]:
# Build the final meeting minutes

def build_minutes(transcript, source_name):
    records = parse_transcript(transcript)

    decisions = extract_decisions(records)
    actions = extract_actions(records)
    dates = extract_dates(records)
    key_points = extract_key_points(records, decisions)

    key_points = [
        normalize_keypoint_output(item)
        for item in dedupe(key_points, threshold=0.78)
    ]

    lines = [
        "MEETING MINUTES",
        "=" * 60,
        f"Source: {source_name}",
        "",
        "KEY POINTS",
        "-" * 20
    ]

    lines.extend(f"• {item}" for item in key_points)

    if not key_points:
        lines.append("• No clear factual key points identified.")

    lines += [
        "",
        "DECISIONS",
        "-" * 20
    ]

    lines.extend(f"• {item}" for item in decisions)

    if not decisions:
        lines.append("• No explicit decisions identified.")

    lines += [
        "",
        "ACTION ITEMS",
        "-" * 20
    ]

    lines.extend(f"• {item}" for item in actions)

    if not actions:
        lines.append("• No explicit action commitments identified.")

    lines += [
        "",
        "IMPORTANT DATES / DEADLINES",
        "-" * 30
    ]

    lines.extend(f"• {item}" for item in dates)

    if not dates:
        lines.append("• No explicit dates or deadlines identified.")

    return "\n".join(lines) + "\n"


In [12]:

generated_files = []

for transcript_path in transcript_files:

    source_name = os.path.basename(
        transcript_path
    )

    base_name = os.path.splitext(
        source_name
    )[0]

    output_path = os.path.join(
        SUMMARY_DIR,
        base_name + "_summary.txt"
    )

    with open(
        transcript_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:
        transcript = f.read()

    minutes = build_minutes(
        transcript,
        source_name
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(minutes)

    generated_files.append(output_path)

    print(
        "OVERWRITTEN:",
        os.path.basename(output_path)
    )

print()
print("=" * 60)
print("FINAL v7 GENERATION COMPLETE")
print("=" * 60)
print(
    "Summary files written:",
    len(generated_files)
)


OVERWRITTEN: meeting_conversation_1_summary.txt
OVERWRITTEN: meeting_conversation_10_summary.txt
OVERWRITTEN: meeting_conversation_11_summary.txt
OVERWRITTEN: meeting_conversation_2_summary.txt
OVERWRITTEN: meeting_conversation_3_summary.txt
OVERWRITTEN: meeting_conversation_4_summary.txt
OVERWRITTEN: meeting_conversation_5_summary.txt
OVERWRITTEN: meeting_conversation_6_summary.txt
OVERWRITTEN: meeting_conversation_7_summary.txt
OVERWRITTEN: meeting_conversation_8_summary.txt
OVERWRITTEN: meeting_conversation_9_summary.txt

FINAL v7 GENERATION COMPLETE
Summary files written: 11


In [13]:

for output_path in generated_files:

    with open(
        output_path,
        "r",
        encoding="utf-8"
    ) as f:
        content = f.read()

    print("\n" + "=" * 75)
    print(os.path.basename(output_path))
    print("=" * 75)
    print(content)



meeting_conversation_1_summary.txt
MEETING MINUTES
Source: meeting_conversation_1.txt

KEY POINTS
--------------------
• To summarize, we are using filtered exports instead of exporting the complete customer database. Segmentation will run only after successful ingestion. The active-customer definition will use active subscriptions. Performance testing will use 15,000, 100,000, and 500,000 transaction levels.
• The API sends the selected values correctly, but the dashboard query only uses the first value.
• I checked those yesterday. Most were caused by missing customer IDs and invalid date formats. I added validation rules for both cases, but there are still a few records that need manual review.
• The dashboard is correct if it filters by active subscription.
• I will run the performance test and record page load times for 15,000, 100,000, and 500,000 transactions.

DECISIONS
--------------------
• No explicit decisions identified.

ACTION ITEMS
--------------------
• Aarav — upload


Run:

**Kernel → Restart Kernel and Run All**

Then inspect the generated files in:

`outputs/Flan-T5/`

This notebook does not retrain the model. It loads `flan_t5_model.pt` created during the training stage.

It does not modify the audio dataset and does not create additional folders.
